# Chest X-ray CNN — From Scratch + MobileNetV2 Fine-Tuning

Dataset: [Kaggle `paultimothymooney/chest-xray-pneumonia`](https://www.kaggle.com/datasets/paultimothymooney/chest-xray-pneumonia) (~2 GB).
Place the extracted folders at `../data/xray/{train,test,val}/{NORMAL,PNEUMONIA}/`.

If the data folder is missing, this notebook trains on the synthetic patches from `scripts/train_xray.py` so it still runs.

In [ ]:
import os, numpy as np, matplotlib.pyplot as plt, tensorflow as tf
from tensorflow.keras import layers, models, optimizers, callbacks
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mn_pre
print('TF', tf.__version__)
%matplotlib inline

## Load data

In [ ]:
DATA = '../data/xray'
IMG = 128
if os.path.isdir(os.path.join(DATA, 'train')):
    train_ds = tf.keras.utils.image_dataset_from_directory(os.path.join(DATA, 'train'), image_size=(IMG, IMG), batch_size=32, color_mode='grayscale')
    val_ds   = tf.keras.utils.image_dataset_from_directory(os.path.join(DATA, 'val'),   image_size=(IMG, IMG), batch_size=32, color_mode='grayscale')
    test_ds  = tf.keras.utils.image_dataset_from_directory(os.path.join(DATA, 'test'),  image_size=(IMG, IMG), batch_size=32, color_mode='grayscale')
    class_names = train_ds.class_names
    USE_REAL = True
else:
    print('Real X-ray data not found — using synthetic.')
    import sys; sys.path.insert(0, '../scripts')
    from train_xray import synth_images
    X, y = synth_images(n_per_class=600)
    # upsample synth to IMG x IMG
    X = tf.image.resize(X, (IMG, IMG)).numpy()
    split = int(0.8*len(X))
    train_ds = tf.data.Dataset.from_tensor_slices((X[:split]*255, y[:split])).shuffle(1024).batch(32)
    val_ds = tf.data.Dataset.from_tensor_slices((X[split:]*255, y[split:])).batch(32)
    test_ds = val_ds
    class_names = ['NORMAL', 'PNEUMONIA']
    USE_REAL = False
print('classes:', class_names)

## Visualise samples

In [ ]:
for imgs, lbls in train_ds.take(1):
    fig, axes = plt.subplots(2, 4, figsize=(10,5))
    for ax, im, l in zip(axes.flat, imgs, lbls):
        ax.imshow(np.array(im).squeeze(), cmap='gray'); ax.set_title(class_names[int(l)]); ax.axis('off')
    plt.show()

## Model A: small CNN from scratch

In [ ]:
def make_scratch():
    m = models.Sequential([
        layers.Input((IMG, IMG, 1)),
        layers.Rescaling(1./255),
        layers.RandomFlip('horizontal'), layers.RandomRotation(0.05),
        layers.Conv2D(32, 3, activation='relu', padding='same'), layers.MaxPool2D(),
        layers.Conv2D(64, 3, activation='relu', padding='same'), layers.MaxPool2D(),
        layers.Conv2D(128, 3, activation='relu', padding='same'), layers.MaxPool2D(),
        layers.Conv2D(128, 3, activation='relu', padding='same'),
        layers.GlobalAveragePooling2D(),
        layers.Dropout(0.3),
        layers.Dense(64, activation='relu'),
        layers.Dense(len(class_names), activation='softmax'),
    ])
    m.compile(optimizer=optimizers.Adam(1e-3), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return m
scratch = make_scratch(); scratch.summary()

In [ ]:
es = callbacks.EarlyStopping(patience=3, restore_best_weights=True)
hist = scratch.fit(train_ds, validation_data=val_ds, epochs=15, callbacks=[es])

In [ ]:
plt.plot(hist.history['accuracy'], label='train'); plt.plot(hist.history['val_accuracy'], label='val')
plt.title('Scratch CNN accuracy'); plt.legend(); plt.show()

## Model B: MobileNetV2 transfer learning + fine-tune

MobileNetV2 needs RGB input (3 channels) at ≥96×96, so we duplicate the gray channel.

In [ ]:
def to_rgb(ds):
    return ds.map(lambda x, y: (tf.image.grayscale_to_rgb(tf.image.resize(x, (96, 96))), y))
train_rgb, val_rgb = to_rgb(train_ds), to_rgb(val_ds)
base = MobileNetV2(input_shape=(96,96,3), include_top=False, weights='imagenet')
base.trainable = False
tl = models.Sequential([
    layers.Input((96,96,3)), layers.Lambda(mn_pre), base,
    layers.GlobalAveragePooling2D(), layers.Dropout(0.3),
    layers.Dense(len(class_names), activation='softmax'),
])
tl.compile(optimizer=optimizers.Adam(1e-3), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
tl.fit(train_rgb, validation_data=val_rgb, epochs=5, callbacks=[es])

In [ ]:
# Unfreeze top layers and fine-tune at a lower LR
base.trainable = True
for l in base.layers[:-30]: l.trainable = False
tl.compile(optimizer=optimizers.Adam(1e-5), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
tl.fit(train_rgb, validation_data=val_rgb, epochs=5, callbacks=[es])

## Save the better model in the format the backend expects

The backend loads `xray_cnn.h5` and expects (64,64,1) grayscale input. Re-export whichever CNN performed better, resized into that shape, so the API stays a drop-in upgrade.

In [ ]:
os.makedirs('../models', exist_ok=True)
scratch.save('../models/xray_cnn.h5')   # adjust if you'd rather export the TL model
print('saved.')